# Mental Health RAG Implementation with LangChain
This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) pipeline using LangChain. We will load mental health coping strategies into a FAISS vector database and use Google's Gemini model to answer questions based on that context.

In [ ]:
import os
import getpass

# Set your Google API Key
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Disable symlink warnings on Windows for HuggingFace
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# 1. Load the data
loader = TextLoader("data/sample_mental_health_docs.txt")
documents = loader.load()

# 2. Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# 3. Create the embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

# 4. Create the retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store created and retriever initialized!")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

# 2. Create the Prompt Template for the chain
system_prompt = (
    "You are an empathetic and professional mental health assistant. "
    "Use the following retrieved context to answer the user's question. "
    "If you don't know the answer or the context doesn't provide enough information, "
    "offer general support but state that you don't have specific advice for that. "
    "Never provide medical diagnoses.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])


question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
print("RAG Chain successfully initialized!")

In [ ]:

user_query = "I've been feeling really overwhelmed and anxious lately. What can I do?"

print(f"User: {user_query}\n")
response = rag_chain.invoke({"input": user_query})

print("Assistant:")
print(response["answer"])

print("\n\n--- Retrieved Source Documents ---")
for i, doc in enumerate(response["context"]):
    print(f"\nDocument {i+1}:\n{doc.page_content}")